In [3]:
!pip install rasterio pandas numpy pyarrow openpyxl tqdm -q

In [4]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# GEBCO_2024 Cloud Optimized GeoTIFF
# ============================================================

RASTER_URL = "https://data.source.coop/alexgleith/gebco-2024/GEBCO_2024.tif"

OUTPUT_RAW = Path("gebco_bathymetry_data/raw")
OUTPUT_PROCESSED = Path("gebco_bathymetry_data/processed")

OUTPUT_RAW.mkdir(parents=True, exist_ok=True)
OUTPUT_PROCESSED.mkdir(parents=True, exist_ok=True)

REGIONS = {
    "bob": {
        "name": "Bay_of_Bengal",
        "west": 80,
        "east": 98,
        "south": 5,
        "north": 22,
        "output_tif": "GEBCO_2024_Bay_of_Bengal.tif"
    },
    "as": {
        "name": "Arabian_Sea",
        "west": 60,
        "east": 75,
        "south": 5,
        "north": 22,
        "output_tif": "GEBCO_2024_Arabian_Sea.tif"
    }
}


def crop_gebco_region(region_key):
    region = REGIONS[region_key]
    output_file = OUTPUT_RAW / region["output_tif"]

    if output_file.exists():
        print("Already exists, skipping:", output_file)
        return output_file

    print("\nCropping:", region["name"])

    with rasterio.open(RASTER_URL) as src:
        window = from_bounds(
            region["west"],
            region["south"],
            region["east"],
            region["north"],
            transform=src.transform
        )

        window = window.round_offsets().round_lengths()

        data = src.read(1, window=window)

        profile = src.profile.copy()
        profile.update({
            "height": data.shape[0],
            "width": data.shape[1],
            "transform": window_transform(window, src.transform),
            "driver": "GTiff",
            "compress": "deflate"
        })

        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(data, 1)

    print("Saved:", output_file)
    print("Size MB:", round(output_file.stat().st_size / (1024 * 1024), 2))

    return output_file


bob_file = crop_gebco_region("bob")
as_file = crop_gebco_region("as")


Cropping: Bay_of_Bengal
Saved: gebco_bathymetry_data/raw/GEBCO_2024_Bay_of_Bengal.tif
Size MB: 13.98

Cropping: Arabian_Sea
Saved: gebco_bathymetry_data/raw/GEBCO_2024_Arabian_Sea.tif
Size MB: 11.63


In [5]:
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("gebco_bathymetry_data/raw")
OUTPUT_DIR = Path("gebco_bathymetry_data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def infer_basin_from_filename(filename):
    filename = filename.lower()

    if "bay_of_bengal" in filename or "bob" in filename:
        return "Bay_of_Bengal"
    elif "arabian_sea" in filename or "_as" in filename:
        return "Arabian_Sea"
    else:
        return "Unknown"


def to_025_degree_grid(values):
    return np.floor(values / 0.25) * 0.25 + 0.125


def classify_ocean_depth(depth_m):
    if pd.isna(depth_m):
        return "unknown"
    elif depth_m < 200:
        return "coastal_shelf"
    elif depth_m < 1000:
        return "continental_slope"
    elif depth_m < 3000:
        return "deep_ocean"
    else:
        return "abyssal_deep_ocean"


def process_one_gebco_tif(file_path):
    file_path = Path(file_path)
    basin = infer_basin_from_filename(file_path.name)

    print("\nProcessing:", file_path.name)

    with rasterio.open(file_path) as src:
        elevation = src.read(1).astype("float32")
        transform = src.transform
        nodata = src.nodata

        if nodata is not None:
            elevation[elevation == nodata] = np.nan

        rows, cols = np.indices(elevation.shape)

        xs, ys = rasterio.transform.xy(
            transform,
            rows,
            cols,
            offset="center"
        )

        lon = np.array(xs)
        lat = np.array(ys)

    df = pd.DataFrame({
        "basin": basin,
        "latitude": lat.ravel(),
        "longitude": lon.ravel(),
        "elevation_m": elevation.ravel()
    })

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["elevation_m"])

    # Keep ocean only: GEBCO ocean bathymetry is negative elevation
    df = df[df["elevation_m"] < 0].copy()

    # Convert negative elevation to positive depth
    df["seafloor_depth_m"] = -df["elevation_m"]

    # Convert 15 arc-second GEBCO grid to project 0.25° grid
    df["lat_025"] = to_025_degree_grid(df["latitude"])
    df["lon_025"] = to_025_degree_grid(df["longitude"])

    df_025 = (
        df.groupby(["basin", "lat_025", "lon_025"], as_index=False)
        .agg(
            seafloor_depth_mean_m=("seafloor_depth_m", "mean"),
            seafloor_depth_median_m=("seafloor_depth_m", "median"),
            seafloor_depth_min_m=("seafloor_depth_m", "min"),
            seafloor_depth_max_m=("seafloor_depth_m", "max"),
            valid_gebco_cells=("seafloor_depth_m", "count")
        )
    )

    df_025["ocean_depth_class"] = df_025["seafloor_depth_mean_m"].apply(classify_ocean_depth)

    print("Processed 0.25° rows:", len(df_025))

    return df_025


tif_files = sorted(INPUT_DIR.glob("GEBCO_2024_*.tif"))

frames = []

for f in tif_files:
    frames.append(process_one_gebco_tif(f))

gebco_025 = pd.concat(frames, ignore_index=True)

print("\nFinal GEBCO table shape:", gebco_025.shape)
gebco_025.head()


Processing: GEBCO_2024_Arabian_Sea.tif
Processed 0.25° rows: 3844

Processing: GEBCO_2024_Bay_of_Bengal.tif
Processed 0.25° rows: 4048

Final GEBCO table shape: (7892, 9)


,basin,lat_025,lon_025,seafloor_depth_mean_m,seafloor_depth_median_m,seafloor_depth_min_m,seafloor_depth_max_m,valid_gebco_cells,ocean_depth_class
0,Arabian_Sea,5.125,60.125,3694.066406,3689.0,3064.0,4472.0,3600,abyssal_deep_ocean
1,Arabian_Sea,5.125,60.375,3646.770264,3687.0,3046.0,3963.0,3600,abyssal_deep_ocean
2,Arabian_Sea,5.125,60.625,3424.227539,3418.0,2964.0,3747.0,3600,abyssal_deep_ocean
3,Arabian_Sea,5.125,60.875,3154.195801,3157.0,2828.0,3602.0,3600,abyssal_deep_ocean
4,Arabian_Sea,5.125,61.125,3300.583984,3272.0,2990.0,3782.0,3600,abyssal_deep_ocean


In [6]:
csv_file = OUTPUT_DIR / "gebco_bathymetry_025deg_bob_as.csv"
excel_file = OUTPUT_DIR / "gebco_bathymetry_025deg_bob_as.xlsx"
parquet_file = OUTPUT_DIR / "gebco_bathymetry_025deg_bob_as.parquet"

gebco_025.to_csv(csv_file, index=False)
gebco_025.to_excel(excel_file, index=False)
gebco_025.to_parquet(parquet_file, index=False)

print("Saved CSV:", csv_file)
print("Saved Excel:", excel_file)
print("Saved Parquet:", parquet_file)

Saved CSV: gebco_bathymetry_data/processed/gebco_bathymetry_025deg_bob_as.csv
Saved Excel: gebco_bathymetry_data/processed/gebco_bathymetry_025deg_bob_as.xlsx
Saved Parquet: gebco_bathymetry_data/processed/gebco_bathymetry_025deg_bob_as.parquet


In [7]:
summary = (
    gebco_025.groupby("basin", as_index=False)
    .agg(
        mean_depth_m=("seafloor_depth_mean_m", "mean"),
        median_depth_m=("seafloor_depth_median_m", "median"),
        shallowest_depth_m=("seafloor_depth_min_m", "min"),
        deepest_depth_m=("seafloor_depth_max_m", "max"),
        grid_cells=("seafloor_depth_mean_m", "count")
    )
)

summary_file = OUTPUT_DIR / "gebco_basin_depth_summary.xlsx"
summary.to_excel(summary_file, index=False)

print("Saved summary:", summary_file)
summary

Saved summary: gebco_bathymetry_data/processed/gebco_basin_depth_summary.xlsx


,basin,mean_depth_m,median_depth_m,shallowest_depth_m,deepest_depth_m,grid_cells
0,Arabian_Sea,3240.042236,3725.5,1.0,5201.0,3844
1,Bay_of_Bengal,2248.048584,2672.0,1.0,4714.0,4048


In [8]:
import shutil
from google.colab import files

zip_name = "gebco_bathymetry_bob_as"

shutil.make_archive(zip_name, "zip", "gebco_bathymetry_data")

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>